In [0]:
import uuid

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import (
    col,
    current_timestamp,
    lit,
    monotonically_increasing_id,
    row_number
)
from pyspark.sql.window import Window

CATALOG = "de_prac"
SCHEMA = "day01"
VOLUME = "data-files"

INCOMING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/incoming"

RUN_ID = str(uuid.uuid4())

print("Run ID:", RUN_ID)
print("Incoming path:", INCOMING_PATH)

In [0]:
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("state", StringType(), True),
    StructField("updated_at", StringType(), True)
])

customers_source = (
    spark.read
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(customer_schema)
    .csv(f"{INCOMING_PATH}/customers")
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name")
    )
)

window_spec = Window.orderBy(monotonically_increasing_id())

customers_bronze = (
    customers_source
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn(
        "source_row_number",
        row_number().over(window_spec)
    )
    .withColumn("run_id", lit(RUN_ID))
)

orders_df = (
    spark.read
    .json(f"{INCOMING_PATH}/orders")
)

print("Customer source rows:", customers_source.count())
print("Order source rows:", orders_df.count())

display(customers_bronze.limit(10))

In [0]:
customers_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("de_prac.day01.customers_bronze")

orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("de_prac.day01.orders_bronze")

print("Bronze Delta tables created successfully.")

In [0]:
customer_source_count = customers_source.count()

customer_bronze_count = (
    spark.table("de_prac.day01.customers_bronze")
    .count()
)

order_bronze_count = (
    spark.table("de_prac.day01.orders_bronze")
    .count()
)

if customer_source_count == customer_bronze_count:
    status = "PASS"
else:
    status = "FAIL"

print("Customer source count :", customer_source_count)
print("Customer Bronze count :", customer_bronze_count)
print("Customer count check   :", status)
print("Orders Bronze count    :", order_bronze_count)

display(
    spark.table("de_prac.day01.customers_bronze")
    .limit(10)
)

In [0]:
from datetime import datetime, timezone

source_count = customers_source.count()
bronze_count = spark.table("de_prac.day01.customers_bronze").count()

status = "SUCCESS" if source_count == bronze_count else "FAILURE"

log_data = [(
    RUN_ID,
    datetime.now(timezone.utc),
    "customers",
    source_count,
    bronze_count,
    status
)]

log_columns = [
    "run_id",
    "log_timestamp",
    "dataset",
    "source_count",
    "bronze_count",
    "status"
]

log_df = spark.createDataFrame(log_data, log_columns)

log_df.write \
    .mode("append") \
    .format("delta") \
    .save(f"/Volumes/de_prac/day01/data-files/logs/customer_ingestion_log")

display(log_df)

In [0]:
import csv

TEST_PATH = f"{INCOMING_PATH}/day01_tests"

dbutils.fs.mkdirs(TEST_PATH)

header = "customer_id,customer_name,email,state,updated_at\n"

# Valid file
dbutils.fs.put(
    f"{TEST_PATH}/valid.csv",
    header +
    "1,Alice,alice@example.com,OH,2026-09-24T12:00:00Z\n"
    "2,Bob,bob@example.com,TX,2026-09-24T12:00:00Z\n",
    True
)

# Header-only file
dbutils.fs.put(
    f"{TEST_PATH}/header_only.csv",
    header,
    True
)

# Empty file
dbutils.fs.put(
    f"{TEST_PATH}/empty.csv",
    "",
    True
)

# Duplicate rows
dbutils.fs.put(
    f"{TEST_PATH}/duplicates.csv",
    header +
    "10,John,john@example.com,CA,2026-09-24T12:00:00Z\n"
    "10,John,john@example.com,CA,2026-09-24T12:00:00Z\n",
    True
)

# Extra whitespace
dbutils.fs.put(
    f"{TEST_PATH}/whitespace.csv",
    header +
    '20,"  Sarah  ","  sarah@example.com  "," OH ",2026-09-24T12:00:00Z\n',
    True
)

print("Day 1 test files created.")

In [0]:
from pyspark.sql.utils import AnalysisException

test_results = []

def add_result(test_name, status, details):
    test_results.append((test_name, status, details))

# 1. Valid file
try:
    valid_df = (
        spark.read
        .option("header", "true")
        .schema(customer_schema)
        .csv(f"{TEST_PATH}/valid.csv")
    )

    count = valid_df.count()
    add_result(
        "Valid file",
        "PASS" if count == 2 else "FAIL",
        f"Rows read: {count}"
    )
except Exception as e:
    add_result("Valid file", "FAIL", str(e))


# 2. Header-only file
try:
    header_only_df = (
        spark.read
        .option("header", "true")
        .schema(customer_schema)
        .csv(f"{TEST_PATH}/header_only.csv")
    )

    count = header_only_df.count()
    add_result(
        "Header-only file",
        "PASS" if count == 0 else "FAIL",
        f"Rows read: {count}"
    )
except Exception as e:
    add_result("Header-only file", "FAIL", str(e))


# 3. Empty file
try:
    empty_df = (
        spark.read
        .option("header", "true")
        .schema(customer_schema)
        .csv(f"{TEST_PATH}/empty.csv")
    )

    count = empty_df.count()

    add_result(
        "Empty file",
        "FAIL",
        f"Expected missing-header failure, but rows read: {count}"
    )

except Exception:
    add_result(
        "Empty file",
        "PASS",
        "Empty/malformed file failed as expected"
    )


# 4. Missing file
try:
    missing_df = (
        spark.read
        .option("header", "true")
        .schema(customer_schema)
        .csv(f"{TEST_PATH}/does_not_exist.csv")
    )

    missing_df.count()

    add_result(
        "Missing file",
        "FAIL",
        "Expected missing-file failure"
    )

except Exception:
    add_result(
        "Missing file",
        "PASS",
        "Missing file failed gracefully"
    )


# 5. Duplicate rows
try:
    duplicate_df = (
        spark.read
        .option("header", "true")
        .schema(customer_schema)
        .csv(f"{TEST_PATH}/duplicates.csv")
    )

    total_count = duplicate_df.count()
    distinct_count = duplicate_df.distinct().count()

    add_result(
        "Duplicate rows preserved",
        "PASS" if total_count == 2 and distinct_count == 1 else "FAIL",
        f"Total: {total_count}, Distinct: {distinct_count}"
    )

except Exception as e:
    add_result("Duplicate rows preserved", "FAIL", str(e))


# 6. Whitespace preservation
try:
    whitespace_df = (
        spark.read
        .option("header", "true")
        .schema(customer_schema)
        .csv(f"{TEST_PATH}/whitespace.csv")
    )

    row = whitespace_df.first()

    whitespace_preserved = (
        row["customer_name"].startswith("  ")
        and row["customer_name"].endswith("  ")
        and row["email"].startswith("  ")
        and row["state"].startswith(" ")
    )

    add_result(
        "Whitespace preserved",
        "PASS" if whitespace_preserved else "FAIL",
        str(row)
    )

except Exception as e:
    add_result("Whitespace preserved", "FAIL", str(e))


results_df = spark.createDataFrame(
    test_results,
    ["test_name", "status", "details"]
)

display(results_df)